In [34]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:14px;}
</style>
"""))

<font color="red" size ="6"><b>ch14. 웹 데이터 수집</b></font>
# 1절.BeautifulSoup 과 parser
    (정적 웹크롤링, 공공api사용)
    
 `pip install bs4`아나콘다를 설치하면 자동 설치되는 패키지에 포함
 
- 공식 사이트 : https://www.crummy.com/software/BeautifulSoup/
- Documentation : https://www.crummy.com/software/BeautifulSoup/bs4/doc/

In [35]:
import requests # HTTP 요청 처리하는 lib
# file://==c:
#http://www.
from requests_file import FileAdapter

In [36]:
# 로컬에 있는 파일을 웹요청하듯이 읽어오는 작업 
s=requests.Session()
s.mount("file://", FileAdapter())#file://로 시작하는 url을 어댑터가 처리
response =s.get("file:///ai/lecNote/01_python/data/ch14_sample.html")#c:/ai/lecNote/data/ch14_sample.html
response

<Response [200]>

In [37]:
if response:
    print('해당 url에 접근함')
else:
    print('해당 url에 거부됨')

해당 url에 접근함


In [13]:
response.status_code
#200:정상
#404:없는 페이지 

200

In [38]:
response.content #html의 binary형식의 내용

b'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n  <meta charset="UTF-8">\r\n</head>\r\n<body>\r\n  <h1 class="greeting css" id="text">Hello, CSS</h1>\r\n  <h1 class="css">Hi, CSS</h1>\r\n  <div id="subject">subject \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90 \xec\x95\x88\xec\x9d\x98 \xeb\x82\xb4\xec\x9a\xa9</div>\r\n  <p>CSS \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\x8a\x94 \xeb\x8b\xa4\xec\x96\x91\xed\x95\x9c \xea\xb3\xb3\xec\x97\x90\xec\x84\x9c \xed\x99\x9c\xec\x9a\xa9\xeb\x90\xa9\xeb\x8b\x88\xeb\x8b\xa4</p>\r\n  <div class="contents">\r\n    \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\xa5\xbc \xec\x96\xb4\xeb\x96\xbb\xea\xb2\x8c \xec\x9e\x91\xec\x84\xb1\xed\x95\x98\xeb\x8a\x90\xeb\x83\x90\xec\x97\x90 \xeb\x94\xb0\xeb\x9d\xbc\r\n    <span>\xeb\x8b\xa4\xeb\xa5\xb8<b>\xec\x9a\x94\xec\x86\x8c\xea\xb0\x80 \xeb\xb0\x98\xed\x99\x98</b></span>\xeb\x90\xa9\xeb\x8b\x88\xeb\x8b\xa4\r\n  </div>\r\n  <div>CSS \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\x8a\x94 \xeb\x8b\xa4\xec\x96\x91\xed\x95\x9c \xea\xb3\

In [39]:
print(response.content.decode('utf-8'))

<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
</head>
<body>
  <h1 class="greeting css" id="text">Hello, CSS</h1>
  <h1 class="css">Hi, CSS</h1>
  <div id="subject">subject 선택자 안의 내용</div>
  <p>CSS 선택자는 다양한 곳에서 활용됩니다</p>
  <div class="contents">
    선택자를 어떻게 작성하느냐에 따라
    <span>다른<b>요소가 반환</b></span>됩니다
  </div>
  <div>CSS 선택자는 다양한 곳에 <b>활용</b>됩니다</div>
</body>
</html>


In [17]:
response.text

'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n  <meta charset="UTF-8">\r\n</head>\r\n<body>\r\n  <h1 class="greeting css" id="text">Hello, CSS</h1>\r\n  <h1 class="css">Hi, CSS</h1>\r\n  <div id="subject">subject 선택자 안의 내용</div>\r\n  <p>CSS 선택자는 다양한 곳에서 활용됩니다</p>\r\n  <div class="contents">\r\n    선택자를 어떻게 작성하느냐에 따라\r\n    <span>다른<b>요소가 반환</b></span>됩니다\r\n  </div>\r\n  <div>CSS 선택자는 다양한 곳에 <b>활용</b>됩니다</div>\r\n</body>\r\n</html>'

In [20]:
#html 파싱 객체 
from bs4 import BeautifulSoup
soup =BeautifulSoup(response.text, #response.content,
                   "html.parser")
#soup.__str__()
#soup

In [30]:
#soup.select_one('선택자'): 해당 선택자 처음 하나 엘리먼트만 
el=soup.select_one('h1.css')
print('el=', el)
print('el.text   =>', el.text)
print('el.string =>', el.string)
print('el의 속성들 =>', el.attrs)
print('el의 class 속성 =>', el.attrs['class'])
print('el의 class 속성 =>', el.attrs.get('class'))
print('el의 href속성 =>', el.attrs.get('href'))
print('el의 name=>', el.name)

el= <h1 class="greeting css" id="text">Hello, CSS</h1>
el.text   => Hello, CSS
el.string => Hello, CSS
el의 속성들 => {'class': ['greeting', 'css'], 'id': 'text'}
el의 class 속성 => ['greeting', 'css']
el의 class 속성 => ['greeting', 'css']
el의 href속성 => None
el의 name=> h1


In [40]:
#2. soup.select_one('선택자'): 해당 선택자 엘리먼트 다 배열로
els = soup.select('h1.css')
print('els=>', els)
print('els들의 text =>',[el.text for el in els])
print('els들의 string =>',[el.string for el in els])
print('els들의 속성들 =>',[el.attrs for el in els])
print('els들의 class 속성 =>',[el.attrs.get('class') for el in els])


els=> [<h1 class="greeting css" id="text">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>]
els들의 text => ['Hello, CSS', 'Hi, CSS']
els들의 string => ['Hello, CSS', 'Hi, CSS']
els들의 속성들 => [{'class': ['greeting', 'css'], 'id': 'text'}, {'class': ['css']}]
els들의 class 속성 => [['greeting', 'css'], ['css']]


In [43]:
#3.soup.find('태그, 속성') vs soup.select_one('선택자'): 해당 속성을 갖고있는 태그 처음 하나만
print('selct_one:', soup.select_one('h1.css'))
print('find     :', soup.find('h1', {'class':'css'}))
print('find     :', soup.find('h1', class_='css'))
print()
print('select_one:',soup.select_one('h1#text'))
print('select_one:', soup.find('h1', {'id':'text'}))

selct_one: <h1 class="greeting css" id="text">Hello, CSS</h1>
find     : <h1 class="greeting css" id="text">Hello, CSS</h1>
find     : <h1 class="greeting css" id="text">Hello, CSS</h1>

select_one: <h1 class="greeting css" id="text">Hello, CSS</h1>
select_one: <h1 class="greeting css" id="text">Hello, CSS</h1>


In [50]:
#4. soup.find_all(태그, 속성)vs soup.select ('선택자') :해당 엘리먼트 다 list로
#print('모든 h1.css와 span태그:', soup.select('h1.css, span'))
print('모든 h1.css.greeting와 span태그:', soup.select('h1.css.greeting, span'))
print('모든 h1.css.greeting와 span태그:', soup.find_all(['h1'],class_='css') +
                                        soup.find_all('span'))

모든 h1.css.greeting와 span태그: [<h1 class="greeting css" id="text">Hello, CSS</h1>, <span>다른<b>요소가 반환</b></span>]
모든 h1.css.greeting와 span태그: [<h1 class="greeting css" id="text">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>, <span>다른<b>요소가 반환</b></span>]


In [53]:
# 없는 엘리먼트 찾기 
print('find_all(빈list):', soup.find_all('a'))
print('find(None)      :', soup.find('a'))
print('select(빈list)   :', soup.select('a'))
print('select_one(None):', soup.select_one('a'))

find_all(빈list): []
find(None)      : None
select(빈list)   : []
select_one(None): None


# 2절. 정적 웹 데이터 수집(정적 웹크롤링)
## 2.1 BeautifulSoup모듈을 활용한 html 웹데이터 수집
### 1)환율정보 가져오기 (네이버증권> 시장지표)

- https://finance.naver.com/marketindex/

    * 크롤링 허용범위는 사이트마다 ~/robots.txt에서 확인할 수 있음
        - Allow : 크롤링 허용가능한 폴더 
        - Disallow:크롤링 제한 폴더

In [60]:
# soup 객체 만드는 방법1
import requests
from bs4 import BeautifulSoup
url='https://finance.naver.com/marketindex/'
response=requests.get(url)
response #response 객체 
print(response.status_code)
#print(response.text) #response.content
soup=BeautifulSoup(response.text, 'html.parser')


200


In [66]:
# soup 객체 만드는 방법2
from urllib.request import urlopen
url='https://finance.naver.com/marketindex/'
response=urlopen(url)
response #HTTPResponse
print(response.status)
#print(response.read().decode('cp949'))
soup =BeautifulSoup(response, 'html.parser')

200


In [77]:
p='1,417,000.70'
float(p.replace(',',''))#방법1
float(''.join(p.split(','))) #방법2

1417000.7

In [82]:
# div.head_info 밑의 span.value(find계열)
prices=[]
headinfos=soup.find_all('div',class_='head_info')
for headinfo in headinfos:
    #print(headinfo)
    price=headinfo.find('span',class_='value')
    prices.append(float(''.join((price.text.split(',')))))
print (prices)
    


[1417.7, 889.17, 1635.6, 210.09, 159.24, 1.1541, 1.3507, 99.71, 83.2, 1863.92, 4441.1, 200855.05]


In [83]:
# span.value(find 계열)
price_els=soup.find_all('span', class_='value')
prices=[round(float(price.text.replace(',','')),1)for price in price_els]
print(prices)

[1417.7, 889.2, 1635.6, 210.1, 159.2, 1.2, 1.4, 99.7, 83.2, 1863.9, 4441.1, 200855.0]


In [86]:
# div.head_info 밑의 span.value
price_els=soup.select('div.head_info > span.value')
#price_els

In [114]:
#타이틀
title_els = soup.select('h3.h_lst > span.blind')
# title_els

In [106]:
# 단위들: div.head_info span.blind
unit_els=soup.select('div.head_info>span>span.blind')
len(unit_els)
units = [unit_el.string for unit_el in unit_els]
units.insert(7,'')
units

['원', '원', '원', '원', '엔', '달러', '달러', '', '달러', '원', '달러', '원']

In [110]:
#상승/하락 div.head_info span.blind
trend_els=soup.select('div.head_info > span.blind')
# trend_els


In [115]:
len(title_els), len(price_els),len(units),len(trend_els)

(12, 12, 12, 12)

In [118]:
for title,price,unit,trend in zip (title_els,price_els,units,trend_els):
        print("{}:{}{}-{}".format(title.text, price.text, unit, trend.text))

미국 USD:1,417.70원-상승
일본 JPY(100엔):889.17원-상승
유럽연합 EUR:1,635.60원-상승
중국 CNY:210.09원-상승
달러/일본 엔:159.2400엔-상승
유로/달러:1.1541달러-하락
영국 파운드/달러:1.3507달러-하락
달러인덱스:99.7100-상승
WTI:83.2달러-상승
휘발유:1863.92원-하락
국제 금:4441.1달러-상승
국내 금:200855.05원-상승


In [132]:
import pandas as pd
data =[]
for title,price,unit,trend in zip (title_els,price_els,units,trend_els):
        data.append({'title':title.text,
                     'price':float(price.text.replace(',','')),
                     'unit':unit,
                     'trend':trend.text})
pd.DataFrame(data)#.to_csv('data/file.csv',index=False)
data

[{'title': '미국 USD', 'price': 1417.7, 'unit': '원', 'trend': '상승'},
 {'title': '일본 JPY(100엔)', 'price': 889.17, 'unit': '원', 'trend': '상승'},
 {'title': '유럽연합 EUR', 'price': 1635.6, 'unit': '원', 'trend': '상승'},
 {'title': '중국 CNY', 'price': 210.09, 'unit': '원', 'trend': '상승'},
 {'title': '달러/일본 엔', 'price': 159.24, 'unit': '엔', 'trend': '상승'},
 {'title': '유로/달러', 'price': 1.1541, 'unit': '달러', 'trend': '하락'},
 {'title': '영국 파운드/달러', 'price': 1.3507, 'unit': '달러', 'trend': '하락'},
 {'title': '달러인덱스', 'price': 99.71, 'unit': '', 'trend': '상승'},
 {'title': 'WTI', 'price': 83.2, 'unit': '달러', 'trend': '상승'},
 {'title': '휘발유', 'price': 1863.92, 'unit': '원', 'trend': '하락'},
 {'title': '국제 금', 'price': 4441.1, 'unit': '달러', 'trend': '상승'},
 {'title': '국내 금', 'price': 200855.05, 'unit': '원', 'trend': '상승'}]

In [134]:
data=[]
for title,price,unit,trend in zip (title_els,price_els,units,trend_els):
    data.append([title.text,float(price.text.replace(',','')),unit,trend.text])
pd.DataFrame(data, columns=['title','price','unit','trend'])
data

[['미국 USD', 1417.7, '원', '상승'],
 ['일본 JPY(100엔)', 889.17, '원', '상승'],
 ['유럽연합 EUR', 1635.6, '원', '상승'],
 ['중국 CNY', 210.09, '원', '상승'],
 ['달러/일본 엔', 159.24, '엔', '상승'],
 ['유로/달러', 1.1541, '달러', '하락'],
 ['영국 파운드/달러', 1.3507, '달러', '하락'],
 ['달러인덱스', 99.71, '', '상승'],
 ['WTI', 83.2, '달러', '상승'],
 ['휘발유', 1863.92, '원', '하락'],
 ['국제 금', 4441.1, '달러', '상승'],
 ['국내 금', 200855.05, '원', '상승']]

### 2)이번주 로또번호 출력
- 방법2에서 User Agent를 추가하여 soup생성 
- https://search.daum.net/search?w=tot&DA=YZR&t__nil_searchbox=btn&q=lotto(다음에서 lotto검색)
````
        1236회(2026.08.08 추첨)
        당첨번호 [12,18,21,29,34,38]
        보너스 10
````

In [139]:
#방법1
import requests
from bs4 import BeautifulSoup
url='https://search.daum.net/search?w=tot&DA=YZR&t__nil_searchbox=btn&q=lotto'
response=requests.get(url)
print('response의 상태 :',response.status_code)
soup=BeautifulSoup(response.text, 'html.parser')
soup


response의 상태 : 200


<!DOCTYPE html>

<html lang="ko" xmlns="http://www.w3.org/1999/xhtml">
<head profile="http://a9.com/-/spec/opensearch/1.1/">
<meta content="text/html;charset=utf-8" http-equiv="content-Type"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<meta content="off" name="autocomplete"/>
<meta content="always" name="referrer"/>
<meta content="telephone=no,address=no,email=no" name="format-detection">
<meta content="lotto – Daum 검색" property="og:title"/>
<meta content="https://search.daum.net/search?w=tot&amp;DA=YZR&amp;t__nil_searchbox=btn&amp;q=lotto" property="og:url"/>
<meta content="Daum 검색에서 lotto에 대한 최신정보를 찾아보세요." property="og:description"/>
<meta content="https://search1.daumcdn.net/search/statics/common/img/2026/daum_og.png" property="og:image"/>
<meta content="다음검색" property="og:site_name"/>
<link href="https://search1.daumcdn.net/search/statics/common/img/2026/0303/favicon.ico" rel="shortcut icon"/>
<link href="//sl.search.daum.net/" rel="preconnect"/>
<link href="//ssl.daum

In [145]:
#방법2
from urllib.request import urlopen, Request
headers={'User-Agent':
         'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
request= Request(url, headers=headers)

response = urlopen(request)
print('response의 상태 :', response.status)
soup=BeautifulSoup(response, 'html.parser')
#soup

response의 상태 : 200


In [160]:
#1236회(2026.08.08 추첨)
#당첨번호 [12,18,21,29,34,38]
#보너스 10
times=soup.select_one('div.prize>strong>span.f_red').text
date= soup.select_one('div.prize>span.date').text
title1=soup.select_one('div.prize > strong').text[-4:]
lotto_numbers=soup.select('div.lottonum > span.ball:nth-last-child(n+3)')
lotto_numbers=soup.select('div.lottonum > span.ball')[:-2]
title2=soup.select_one('div.lottonum span.screen_out').text
bonus_number=soup.select_one('div.lottonum > span.bg_ball1').text
print(times, date)
print(title1,[int(numbers.text) for numbers in lotto_numbers])
print(title2, bonus_number)

1236회 (2026.08.08 추첨)
당첨번호 [12, 18, 21, 29, 34, 38]
보너스 10


In [164]:
# 위의 select계열 함수를 find계열함수로 변경하여 구현해 보기
# times = soup.select_one('div.prize span.f_red').text
prize = soup.find('div', class_='prize')
times = prize.find('span', class_='f_red').text

# date = soup.select_one('div.prize > span.date').text
date = prize.find('span', class_='date').text

# title1 = soup.select_one('div.prize > strong').text[-4:]
title1 = prize.find('strong').text[-4:]

# lotto_numbers = soup.select('div.lottonum > span.ball')[:-2]
lottonum = soup.find('div', class_='lottonum')
lotto_numbers = lottonum.find_all('span', class_='ball')[:-2]

# title2 = soup.select_one('div.lottonum span.screen_out').text
title2 = lottonum.find('span', class_='screen_out').text

# bonus_number = soup.select_one('div.lottonum > span.bg_ball1').text
bonus_number = lottonum.find('span', class_='bg_ball1').text

print(times, date)
print(title1, [int(numbers.text) for numbers in lotto_numbers])
print(title2, bonus_number)

'당첨번호'

### 3)다음 뉴스 검색리스트 
```
no title  href
0 타이틀1  http://~
1 타이틀2  http://~
2 타이틀3  http://~

```

In [166]:
# 방법1
import requests
from bs4 import BeautifulSoup
word ='비트코인'
url=f'https://search.daum.net/search?nil_suggest=btn&w=news&DA=PGD&cluster=y&q={word}'
print(url)
response=requests.get(url)
print(response.status_code)
soup=BeautifulSoup(response.text, 'html.parser')

https://search.daum.net/search?nil_suggest=btn&w=news&DA=PGD&cluster=y&q=비트코인
200


In [173]:
#방법2 
from urllib.request import urlopen, Request
from urllib.parse import quote
word=quote('비트코인')
url=f'https://search.daum.net/search?nil_suggest=btn&w=news&DA=PGD&cluster=y&q={word}'
print(url)
headers={'User-Agent':
         'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
request= Request(url, headers=headers)
request=Request(url)
request.add_header('User-Agent',
                  'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36')
response =urlopen(request)
print(response.status)
soup=BeautifulSoup(response, 'html.parser')
#soup

https://search.daum.net/search?nil_suggest=btn&w=news&DA=PGD&cluster=y&q=%EB%B9%84%ED%8A%B8%EC%BD%94%EC%9D%B8
200


In [185]:
items_find_list=[]#검색한 결과를 담을 dict리스트 
items_el = soup.select('div.item-title> strong.tit-g.clamp-g >a')
for idx, item in enumerate(items_el):
    print(idx, item.text)
    items_find_list.append({'no':idx, 
                           'title':item.text,
                           'link':item.attrs.get('href')})
import pandas as pd
pd.DataFrame(items_find_list)

0  9000만원 깨진 비트코인…美물가지수 앞두고 위험회피 심리↑ 
1  [코인뉴스] 비트코인 박스권 지속…다음 변수는? 
2  [코인뉴스] CPI 앞둔 비트코인…박스권 속 엇갈린 베팅 
3  9000만원대 갇힌 비트코인…'100만弗 vs 4만弗' 극단 전망 
4  [코인시세] 비트코인 6만3천달러대 약세…CPI 경계감 지속 
5  [08:03 가상자산] 비트코인, 美 CPI 결과 발표 앞두고 9000만원 선 붕괴 
6  트럼프 회사, 비트코인에 3300억 베팅했다 '쓴맛'⋯손실 12배 늘었다 
7  금리 공포 걷히자 "금값 조정 끝났다"… 비트코인은 나 홀로 약세 
8  [아주경제 코이너스 브리핑] 호르무즈 불확실성에…비트코인 6만3000달러대 횡보 
9  비트코인 다시 6.3만달러대로…美 CPI 발표 ‘촉각’ [코인 모닝콜] 


,no,title,link
0,0,9000만원 깨진 비트코인…美물가지수 앞두고 위험회피 심리↑,http://v.daum.net/v/20260812104759348
1,1,[코인뉴스] 비트코인 박스권 지속…다음 변수는?,http://v.daum.net/v/20260812093050056
2,2,[코인뉴스] CPI 앞둔 비트코인…박스권 속 엇갈린 베팅,http://v.daum.net/v/20260812163039451
3,3,9000만원대 갇힌 비트코인…'100만弗 vs 4만弗' 극단 전망,http://v.daum.net/v/20260812161211645
4,4,[코인시세] 비트코인 6만3천달러대 약세…CPI 경계감 지속,http://v.daum.net/v/20260812103841887
5,5,"[08:03 가상자산] 비트코인, 美 CPI 결과 발표 앞두고 9000만원 선 붕괴",http://v.daum.net/v/20260812080815138
6,6,"트럼프 회사, 비트코인에 3300억 베팅했다 '쓴맛'⋯손실 12배 늘었다",http://v.daum.net/v/20260811144238992
7,7,"금리 공포 걷히자 ""금값 조정 끝났다""… 비트코인은 나 홀로 약세",http://v.daum.net/v/20260812164709172
8,8,[아주경제 코이너스 브리핑] 호르무즈 불확실성에…비트코인 6만3000달러대 횡보,http://v.daum.net/v/20260812082712570
9,9,비트코인 다시 6.3만달러대로…美 CPI 발표 ‘촉각’ [코인 모닝콜],http://v.daum.net/v/20260812082137419


In [184]:
items_find_list = [] #검색한 결과를 담을 2차원 리스트 
items_el=soup.select('div.item-title > strong.tit-g > a')
for idx, item in enumerate(items_el):
    items_find_list.append([idx, item.text, item.attrs.get('href')])
pd.DataFrame(items_find_list, columns=['순번', '기사제목','링크'])

,순번,기사제목,링크
0,0,9000만원 깨진 비트코인…美물가지수 앞두고 위험회피 심리↑,http://v.daum.net/v/20260812104759348
1,1,[코인뉴스] 비트코인 박스권 지속…다음 변수는?,http://v.daum.net/v/20260812093050056
2,2,[코인뉴스] CPI 앞둔 비트코인…박스권 속 엇갈린 베팅,http://v.daum.net/v/20260812163039451
3,3,9000만원대 갇힌 비트코인…'100만弗 vs 4만弗' 극단 전망,http://v.daum.net/v/20260812161211645
4,4,[코인시세] 비트코인 6만3천달러대 약세…CPI 경계감 지속,http://v.daum.net/v/20260812103841887
5,5,"[08:03 가상자산] 비트코인, 美 CPI 결과 발표 앞두고 9000만원 선 붕괴",http://v.daum.net/v/20260812080815138
6,6,"트럼프 회사, 비트코인에 3300억 베팅했다 '쓴맛'⋯손실 12배 늘었다",http://v.daum.net/v/20260811144238992
7,7,"금리 공포 걷히자 ""금값 조정 끝났다""… 비트코인은 나 홀로 약세",http://v.daum.net/v/20260812164709172
8,8,[아주경제 코이너스 브리핑] 호르무즈 불확실성에…비트코인 6만3000달러대 횡보,http://v.daum.net/v/20260812082712570
9,9,비트코인 다시 6.3만달러대로…美 CPI 발표 ‘촉각’ [코인 모닝콜],http://v.daum.net/v/20260812082137419


In [190]:
# 다음 뉴스 검색 함수 (원하는 키워드, 원하는 페이지로)
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
def collect_list(keyword, page):
    'keyword로 해당 page에 검색한 결과 dict list return'
    #url=f'https://search.daum.net/search?nil_suggest=btn&w=news&DA=PGD&cluster=y&q={keyword}&p={page}'
    url = 'https://search.daum.net/search?w=news'
    params={'q':keyword, 'p':page}
    response=request.get(url, params=params)
    soup = BeautifulSoup(response.text, 'html.parser')
    items_el=soup.select('div.item-title > strong.tit-g > a')
    for idx, item in enumerate(items_el):
        items_find_list.append({'no':(page-1)*10+idx,'title':item.text,'link':item.attrs.get('href')})
        return items_find_list

In [191]:
collect_list('꽃',2)

AttributeError: 'Request' object has no attribute 'get'